In [10]:
import numpy as np
import pandas as pd
import ast
import os
import re

# =====================================================
# CONFIG
# =====================================================
FILES = [
    "cpd_kappa_experiment_old.csv",
    "cpd_kappa_experiment_new.csv"
]

TRUE_CP = [100, 200]
TOL = 10
T = 296


# =====================================================
# parse est_CP（完全照搬）
# =====================================================
def parse_cp(x):
    if pd.isna(x):
        return []

    s = str(x).strip()

    if s in ["", "nan", "None"]:
        return []

    s = re.sub(r"np\.int64\((\d+)\)", r"\1", s)

    if "[" in s and "]" in s and "," not in s:
        s = s.replace(" ", ",")

    try:
        return list(ast.literal_eval(s))
    except:
        return []


# =====================================================
# evaluation（完全不动）
# =====================================================
def evaluate_cp(est_cp, true_cp, tol, T):
    est_cp = np.array(est_cp, dtype=int)
    true_cp = np.array(true_cp, dtype=int)

    if len(est_cp) == 0:
        FP = 0
        FN = len(true_cp)
        Dt_e = np.inf
        De_t = np.inf
        CE = abs(len(est_cp) - len(true_cp))
        CS = 0.0
        return FP, FN, Dt_e, De_t, CE, CS

    dist_mat = np.abs(est_cp[:, None] - true_cp[None, :])

    min_dist_true = dist_mat.min(axis=0)
    min_dist_est  = dist_mat.min(axis=1)

    FP = np.sum(min_dist_est > tol)
    FN = np.sum(min_dist_true > tol)

    Dt_e = np.max(min_dist_true)
    De_t = np.max(min_dist_est)

    CE = abs(len(est_cp) - len(true_cp))

    def get_segments(cp, T):
        cp = np.sort(cp)
        segs = []
        prev = 0
        for c in cp:
            segs.append(set(range(prev, c)))
            prev = c
        segs.append(set(range(prev, T)))
        return segs

    G  = get_segments(true_cp, T)
    Gp = get_segments(est_cp, T)

    def jaccard(A, B):
        return len(A & B) / len(A | B) if len(A | B) > 0 else 0

    CS = 0
    for A in G:
        best = max(jaccard(A, Ap) for Ap in Gp)
        CS += len(A) * best
    CS /= T

    return FP, FN, Dt_e, De_t, CE, CS


# =====================================================
# LOAD FILES
# =====================================================
df_list = []

for f in FILES:
    if not os.path.exists(f):
        continue

    tmp = pd.read_csv(f)
    tmp["source"] = "old" if "old" in f else "new"
    tmp["est_CP"] = tmp["est_CP"].apply(parse_cp)

    df_list.append(tmp)

df = pd.concat(df_list, ignore_index=True)

df["kappa"] = df["kappa"].astype(float)


# =====================================================
# COMPUTE METRICS
# =====================================================
df = df.drop(columns=["CE"], errors="ignore")

metrics = df["est_CP"].apply(
    lambda cp: evaluate_cp(cp, TRUE_CP, TOL, T)
)

metrics_df = pd.DataFrame(
    metrics.tolist(),
    columns=["FP", "FN", "Dt_e", "De_t", "CE", "CS"]
)

df = pd.concat([df, metrics_df], axis=1)
df["rho"] = df["kappa"]
df = df.drop(columns=["kappa"])

metrics_cols = ["FP","FN","Dt_e","De_t","CE","CS"]

summary_mean = (
    df.groupby("rho")[metrics_cols]
    .mean()
)

summary_sd = (
    df.groupby("rho")[metrics_cols]
    .std()
)

summary = summary_mean.copy()

for m in metrics_cols:
    summary[f"{m}_sd"] = summary_sd[m]

summary = summary.reset_index().round(3)
summary = summary.sort_values("rho")

def to_latex_table(summary):

    best = {
        "FP": summary["FP"].min(),
        "FN": summary["FN"].min(),
        "Dt_e": summary["Dt_e"].min(),
        "De_t": summary["De_t"].min(),
        "CE": summary["CE"].min(),
        "CS": summary["CS"].max(),
    }

    def bold(val, best_val):
        if np.isclose(val, best_val):
            return f"\\textbf{{{val:.3f}}}"
        return f"{val:.3f}"

    latex = []
    latex.append("\\begin{table}[!ht]")
    latex.append("\\caption{Sensitivity analysis with respect to $\\rho$.}")
    latex.append("\\label{tab:rho}")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append("\\begin{tabular}{lcccccc}")
    latex.append("\\toprule")
    latex.append("$\\rho$ & FP$\\downarrow$ & FN$\\downarrow$ & $D_{t\\to e}$ & $D_{e\\to t}$ & CE$\\downarrow$ & CS$\\uparrow$ \\\\")
    latex.append("\\midrule")

    for _, r in summary.iterrows():

        # ===== mean 行 =====
        mean_row = [
            f"{r['rho']:.1f}",
            bold(r["FP"], best["FP"]),
            bold(r["FN"], best["FN"]),
            bold(r["Dt_e"], best["Dt_e"]),
            bold(r["De_t"], best["De_t"]),
            bold(r["CE"], best["CE"]),
            bold(r["CS"], best["CS"]),
        ]

        latex.append(
            " & ".join(mean_row) + " \\\\"
        )

        # ===== sd 行 =====
        sd_row = [
            "",
            f"({r['FP_sd']:.3f})",
            f"({r['FN_sd']:.3f})",
            f"({r['Dt_e_sd']:.3f})",
            f"({r['De_t_sd']:.3f})",
            f"({r['CE_sd']:.3f})",
            f"({r['CS_sd']:.3f})",
        ]

        latex.append(
            " & ".join(sd_row) + " \\\\"
        )

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)

print(to_latex_table(summary))

\begin{table}[!ht]
\caption{Sensitivity analysis with respect to $\rho$.}
\label{tab:rho}
\centering
\small
\begin{tabular}{lcccccc}
\toprule
$\rho$ & FP$\downarrow$ & FN$\downarrow$ & $D_{t\to e}$ & $D_{e\to t}$ & CE$\downarrow$ & CS$\uparrow$ \\
\midrule
0.2 & \textbf{0.300} & \textbf{0.000} & \textbf{0.000} & \textbf{12.600} & \textbf{0.300} & \textbf{0.964} \\
 & (0.483) & (0.000) & (0.000) & (21.397) & (0.483) & (0.058) \\
0.5 & 0.800 & 0.600 & 34.900 & 36.600 & 0.600 & 0.805 \\
 & (0.632) & (0.516) & (40.297) & (29.281) & (0.516) & (0.121) \\
0.8 & 1.000 & \textbf{0.000} & 0.200 & 61.900 & 1.000 & 0.901 \\
 & (0.471) & (0.000) & (0.632) & (24.637) & (0.471) & (0.053) \\
1.5 & 0.700 & 0.400 & 27.100 & 42.000 & 0.500 & 0.845 \\
 & (0.675) & (0.516) & (39.515) & (36.588) & (0.527) & (0.149) \\
2.0 & 1.000 & 0.400 & 32.400 & 65.400 & 0.800 & 0.790 \\
 & (0.471) & (0.516) & (42.077) & (23.287) & (0.632) & (0.131) \\
\bottomrule
\end{tabular}
\end{table}
